# 03 · Clasificación de intención con QLoRA · de malo a util

> Coges un modelo moderno, lo pones a hacer algo util (routing de mensajes de soporte), ves que
> zero-shot se queda corto, le haces QLoRA con Unsloth, y vuelves a medir: mucho mejor. El mismo arco que el
> Lab 1, pero con el stack de hoy.

## Qué aprendemos

1. **QLoRA + Unsloth** sobre Gemma 4 (E2B/E4B, selector `MODEL_SIZE`) en una T4 gratis.
2. Una tarea **util y medible**: routing de intención (banking77, 77 intents finos).
3. El **antes/después con un número**: accuracy zero-shot -> fine-tuned.

## Por qué importa

Es el caso real de empresa: un modelo generico clasifica regular entre muchas categorías parecidas;
fine-tunearlo en TUS categorías lo vuelve fiable. Y con QLoRA cabe en una T4.

## Conecta con

- **Slides de Fine-tuning · Deck 2** : QLoRA, Unsloth, chat templates, train_on_responses_only.
- **Lab 02** : LoRA en pequeño (GPT-2); aquí QLoRA en modelo moderno.
- **Módulo de Evaluación** : la evaluación (accuracy / F1) se profundiza allí.

## Stack

`unsloth/gemma-4-E2B-it`/`E4B-it` (4-bit, selector `MODEL_SIZE`) · `unsloth` · `trl` · `datasets` · dataset `banking77`.

## Hardware

T4 (Colab Free). Ejecución de referencia (oct-2026, T4): **~40 min en total**. Carga ~3 min, eval zero-shot ~4 min,
entrenamiento ~27 min (en T4 Unsloth entrena Gemma 4 en float32 porque no admite float16) y eval final ~4 min.
Resultado de esa ejecución: accuracy **56% zero-shot -> 85% con QLoRA** (150 mensajes de test).


---

## Setup


In [ ]:
!pip install -qU unsloth datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 5.9 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/43.1 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 41.3/43.1 MB 183.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 43.1/43.1 MB 186.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 43.1/43.1 MB 186.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/423.1 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 24.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.


In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch
from datasets import load_dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


---

# 1 · Cargamos Gemma 4 con Unsloth (4-bit · E2B por defecto)


In [ ]:
# Cambia MODEL_SIZE: "mini" = E2B (T4, rapido) · "big" = E4B (T4 tambien, mejor calidad)
MODEL_SIZE = "mini"
MODELS = {"mini": "unsloth/gemma-4-E2B-it", "big": "unsloth/gemma-4-E4B-it"}

model, tokenizer = FastModel.from_pretrained(
    model_name = MODELS[MODEL_SIZE],
    max_seq_length = 2048,
    load_in_4bit = True,          # QLoRA de fabrica
    full_finetuning = False,
)
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")

==((====))==  Unsloth 2026.9.14: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

---

# 2 · La tarea · routing de intención (banking77)

77 intenciones finas de banca (tarjeta perdida, transferencia pendiente, cambio de PIN...).
Clasificar bien entre tantas categorías parecidas es justo donde un modelo generico flojea.


In [ ]:
# banking77 via el mirror parquet mteb/banking77 · columnas text/label + label_text (nombre de la intencion)
ds = load_dataset("mteb/banking77")
id2label = {row["label"]: row["label_text"] for split in ds for row in ds[split]}
names = [id2label[i] for i in range(len(id2label))]   # las 77 intenciones, en orden de id
LABELS = ", ".join(names)
print(f"{len(names)} intenciones. Ejemplos: {names[:5]} ...")
print("\nEjemplo:")
print(" mensaje:", ds["test"][0]["text"])
print(" intent :", names[ds["test"][0]["label"]])

README.md:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9993 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3076 [00:00<?, ? examples/s]

77 intenciones. Ejemplos: ['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up'] ...

Ejemplo:
 mensaje: How do I locate my card?
 intent : card_arrival


In [5]:
CLS_TEMPLATE = (
    "You are a banking support router. Read the customer message and reply with EXACTLY one "
    "intent label from the list, nothing else.\n"
    "Intents: {labels}\n"
    "Message: {message}\n"
    "Intent:"
)

@torch.no_grad()
def classify(message, max_new_tokens=16):
    prompt = CLS_TEMPLATE.format(labels=LABELS, message=message)
    msgs = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    # Tokenizamos en un paso: vale para tokenizer de texto y para el processor multimodal de Gemma 4
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def parse_label(gen):
    g = gen.strip().lower()
    for name in names:                 # match exacto
        if g == name.lower():
            return name
    for name in names:                 # o que aparezca en la salida
        if name.lower() in g:
            return name
    return g.split("\n")[0].strip()    # fallback

In [6]:
def eval_accuracy(n=150, seed=3407):
    sub = ds["test"].shuffle(seed=seed).select(range(n))
    ok = 0
    for ex in sub:
        pred = parse_label(classify(ex["text"]))
        if pred == names[ex["label"]]:
            ok += 1
    return ok / n

---

# 3 · El "antes" · zero-shot (el modelo base)

Sin entrenar nada, le pedimos que clasifique. Mira un par de ejemplos y la accuracy global.


In [7]:
for ex in ds["test"].select(range(3)):
    print("mensaje:", ex["text"])
    print("  predijo:", parse_label(classify(ex["text"])))
    print("  correcto:", names[ex["label"]], "\n")

acc_base = eval_accuracy()
print(f"Accuracy zero-shot (base): {acc_base:.1%}")

mensaje: How do I locate my card?


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


  predijo: card_arrival
  correcto: card_arrival 

mensaje: I still have not received my new card, I ordered over a week ago.


  predijo: card_arrival
  correcto: card_arrival 

mensaje: I ordered a card but it has not arrived. Help please!


  predijo: card_arrival
  correcto: card_arrival 



Accuracy zero-shot (base): 56.0%


Regular: con 77 intenciones tan parecidas, el modelo generico se confunde y no siempre devuelve
una etiqueta exacta de la lista. Vamos a especializarlo.


---

# 4 · QLoRA · preparamos el adapter y los datos


In [8]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [9]:
# Convertimos cada ejemplo a conversacion (usuario pregunta -> assistant responde la intent)
def to_conv(ex):
    user = CLS_TEMPLATE.format(labels=LABELS, message=ex["text"])
    return {"conversations": [
        {"role": "user",      "content": [{"type": "text", "text": user}]},
        {"role": "assistant", "content": [{"type": "text", "text": names[ex["label"]]}]},
    ]}

train_conv = ds["train"].shuffle(seed=3407).select(range(2000)).map(to_conv)

def apply_template(examples):
    return {"text": tokenizer.apply_chat_template(examples["conversations"], tokenize=False)}

train_text = train_conv.map(apply_template, batched=True)
print(train_text[0]["text"][:400])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

<bos><|turn>user
You are a banking support router. Read the customer message and reply with EXACTLY one intent label from the list, nothing else.
Intents: activate_my_card, age_limit, apple_pay_or_google_pay, atm_support, automatic_top_up, balance_not_updated_after_bank_transfer, balance_not_updated_after_cheque_or_cash_deposit, beneficiary_not_allowed, cancel_transfer, card_about_to_expire, card_


---

# 5 · Entrenamos con Unsloth (QLoRA)


In [10]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_text,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

# Entrenar SOLO sobre la respuesta del assistant (la etiqueta), no sobre el prompt
# Derivamos los marcadores de turno del propio chat template (Gemma 4 usa el formato <|turn>role).
# Asi no dependemos del formato exacto ni de si el rol assistant se renderiza como "model".
_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "U_CONTENT"}, {"role": "assistant", "content": "A_CONTENT"}],
    tokenize=False,
)
instruction_part = _probe[:_probe.index("U_CONTENT")].rstrip("\n").split("\n")[-1] + "\n"
response_part    = _probe[:_probe.index("A_CONTENT")].rstrip("\n").split("\n")[-1] + "\n"
print("Marcadores detectados ->", repr(instruction_part), repr(response_part))

# Entrenar SOLO sobre la respuesta del assistant (la etiqueta), no sobre el prompt
trainer = train_on_responses_only(
    trainer,
    instruction_part = instruction_part,
    response_part = response_part,
)
trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

Marcadores detectados -> '<bos><|turn>user\n' '<|turn>model\n'


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss
20,0.759114
40,0.168181
60,0.167912
80,0.125002
100,0.132615
120,0.110108
140,0.095363
160,0.105604
180,0.079728
200,0.077900


TrainOutput(global_step=250, training_loss=0.15920177626609802, metrics={'train_runtime': 1625.6937, 'train_samples_per_second': 1.23, 'train_steps_per_second': 0.154, 'total_flos': 1.6968858926119296e+16, 'train_loss': 0.15920177626609802, 'epoch': 1.0})

---

# 6 · El "después" · medimos otra vez

Mismos ejemplos, misma métrica. La única diferencia es el QLoRA.


In [11]:
for ex in ds["test"].select(range(3)):
    print("mensaje:", ex["text"])
    print("  predijo:", parse_label(classify(ex["text"])))
    print("  correcto:", names[ex["label"]], "\n")

acc_ft = eval_accuracy()
print(f"Accuracy · zero-shot: {acc_base:.1%}  ->  QLoRA: {acc_ft:.1%}")

mensaje: How do I locate my card?


  predijo: card_arrival
  correcto: card_arrival 

mensaje: I still have not received my new card, I ordered over a week ago.


  predijo: card_arrival
  correcto: card_arrival 

mensaje: I ordered a card but it has not arrived. Help please!


  predijo: card_arrival
  correcto: card_arrival 



Accuracy · zero-shot: 56.0%  ->  QLoRA: 85.3%


Mismo modelo, misma tarea, mismo prompt: el QLoRA lo ha vuelto **fiable** clasificando entre las 77
intenciones. Y todo en una T4, entrenando solo el adapter.


---

# Cierre · lo que nos llevamos

- Cogimos un modelo moderno, vimos que zero-shot **flojeaba** en una tarea util (routing de soporte).
- Con **QLoRA + Unsloth** lo especializamos en TUS categorías y subimos la accuracy de forma medible.
- Mismo patron que el Lab 1 (malo -> entrenar -> mejor), pero con el stack real de 2026.

## Hacia el alignment

Tu modelo ya sabe TAREAS. Pero saber una tarea no es lo mismo que **comportarse** (ser util, honesto,
inofensivo). En los siguientes labs pasamos al alignment: Reward Model + RLHF y luego DPO/ORPO.
